# Train Sunbae on Colab (GPU)

Before running:
1. Runtime -> Change runtime type -> **GPU** (T4 is fine).
2. Upload `dataset/train.bin`, `dataset/val.bin` and `dataset/meta.json`
   to Google Drive: `MyDrive/sunbae/dataset/`.
3. The code must be pushed to GitHub (branch `asadbek`).

The dataset is copied to `/content/dataset/` (fast local disk) before training.
Checkpoints go to `MyDrive/sunbae/checkpoints/`, so they survive a disconnect.
After a disconnect, just run all cells again: training resumes from `last.pt`.

In [3]:
# Check that a GPU is attached
!nvidia-smi

Sun Nov  8 14:47:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# Mount Google Drive (data in, checkpoints out)
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
import os

REPO_URL = "https://github.com/agus787-dev/sunbae"
BRANCH = "asadbek"
REPO_DIR = "/content/sunbae"
DATA_DIR = "/content/drive/MyDrive/sunbae/dataset"   # dataset on Drive
LOCAL_DATA_DIR = "/content/dataset"                  # fast local copy for training
OUT_DIR = "/content/drive/MyDrive/sunbae/checkpoints"  # checkpoints stay on Drive

# Stop early if the dataset is missing on Drive
for name in ["train.bin", "val.bin", "meta.json"]:
    path = os.path.join(DATA_DIR, name)
    assert os.path.exists(path), f"missing {path} - upload dataset/ to Drive first"
    print(f"{name:<10} {os.path.getsize(path) / 1e6:>8.1f} MB")

train.bin     656.6 MB
val.bin         6.5 MB
meta.json       0.0 MB


In [6]:
# Copy the dataset from Drive to the local disk. Training reads many small
# random windows, which is slow on the Drive mount. Skips files already copied.
import shutil

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
for name in ["train.bin", "val.bin", "meta.json"]:
    src, dst = os.path.join(DATA_DIR, name), os.path.join(LOCAL_DATA_DIR, name)
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        print(f"{name:<10} already copied")
    else:
        shutil.copy(src, dst)
        print(f"{name:<10} copied")

train.bin  copied
val.bin    copied
meta.json  copied


In [7]:
# Clone the repo (or update it if this runtime already has it)
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

Cloning into '/content/sunbae'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 121 (delta 44), reused 102 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 276.44 KiB | 12.02 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/sunbae
6e68284 (HEAD -> asadbek, origin/asadbek) Add training loop, checkpoints, Colab notebook and README update


In [8]:
!pip install -q -r requirements.txt

In [9]:
# All 4 checks must pass before spending GPU time
!python tests/test_model.py


Shape test...
  PASS

Random-init loss test...
  random-init loss = 9.076   (expected ~= 8.987)
  PASS

Overfit-one-batch test...
  final loss after 200 steps on one batch = 0.0870
  PASS

Causality test...
  PASS

All checks passed


In [10]:
# Train. If last.pt exists on Drive, resume from it automatically.
last_pt = os.path.join(OUT_DIR, "last.pt")
resume = f"--resume {last_pt}" if os.path.exists(last_pt) else ""
print("Resuming from", last_pt if resume else "nothing - starting a new run")

!python scripts/train.py --config configs/run_01.yaml \
    --data_dir {LOCAL_DATA_DIR} --out_dir {OUT_DIR} {resume}

Resuming from /content/drive/MyDrive/sunbae/checkpoints/last.pt
Device: cuda | precision: torch.bfloat16
Resumed from /content/drive/MyDrive/sunbae/checkpoints/last.pt at step 6751, best val loss 3.1821
=== Training Sunbae: 13,817,856 parameters, 328,321,165 train tokens ===
Tokens per optimizer step: 32,768
Planned total tokens     : 262,144,000
Steps                    : 6751 -> 7999

step  6760 | loss 3.3841 | lr 4.65e-05 | grad_norm 0.64 | tok/s 25,435 | eta 26m
step  6780 | loss 3.3955 | lr 4.60e-05 | grad_norm 0.67 | tok/s 25,428 | eta 26m
step  6800 | loss 3.3972 | lr 4.55e-05 | grad_norm 0.68 | tok/s 24,549 | eta 26m
step  6820 | loss 3.2975 | lr 4.50e-05 | grad_norm 0.64 | tok/s 23,903 | eta 26m
step  6840 | loss 3.2626 | lr 4.45e-05 | grad_norm 0.66 | tok/s 24,529 | eta 25m
step  6860 | loss 3.2963 | lr 4.40e-05 | grad_norm 0.66 | tok/s 24,383 | eta 25m
step  6880 | loss 3.2305 | lr 4.35e-05 | grad_norm 0.66 | tok/s 24,188 | eta 25m
step  6900 | loss 3.3069 | lr 4.30e-05 | gr